# ⚛️ QIP-Bench: What Really Determines the Performance of a Quantum Image-Processing System?

**CSE3728 Quantum Image Processing & Emerging Visual Computing · BML Munjal University · Problem Statement 9**

This Colab notebook is the complete live demonstration of the project. Each section runs **real quantum circuits** on
the Qiskit Aer simulator:

| § | Demonstration | Time |
|---|---|---|
| 0 | Setup: load project, install Qiskit, verify environment | ~2 min |
| 1 | Correctness: 27 automated tests against exact classical answers | 15 s |
| 2 | The quantum pipeline: preprocessing → FRQI/NEQR encoding → measurement → reconstruction | 1 min |
| 3 | Seeing the quantum states: Bloch sphere, pixel ↔ qubit mapping, circuits | 1 min |
| 4 | Wavefunction collapse: rebuilding the image from real shots | 1 min |
| 5 | Factor experiments: noise cliff, shots, resolution, bit depth, circuit structure | 2 min |
| 6 | Quantum edge detection (QHED) | 1 min |
| 7 | Research answer: factor ranking from a 2⁷ factorial study (≈ 2 500 simulated runs) | 1 min |
| 8 | Full web dashboard running inside Colab | 1 min |
| 9 | Live control panel: the faculty can choose any factor | open |

### ▶ How to run
1. **Runtime → Run all** (or `Ctrl+F9`). In §0 choose how to load the project (zip upload, GitHub or Google Drive).
2. No GPU is needed. The default CPU runtime is enough.

---
## 0 · Setup

### 0.1 Load the project
Choose **one** source in the form below:
* **upload**: upload `qip-bench.zip` (the project folder, zipped) when the file picker appears.
* **github**: paste your repository URL.
* **drive**: put the project folder in Google Drive and give its path.

In [ ]:
#@title 0.1 · Load the project { display-mode: "form" }
SOURCE = "upload"  #@param ["upload", "github", "drive"]
GITHUB_URL = "https://github.com/<your-user>/qip-bench.git"  #@param {type:"string"}
DRIVE_PATH = "/content/drive/MyDrive/qip-bench"  #@param {type:"string"}

import os, sys, glob, shutil, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules

def find_project(base):
    hits = sorted(glob.glob(os.path.join(base, "**", "qipbench", "__init__.py"), recursive=True), key=len)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

if IN_COLAB:
    PROJECT = find_project("/content/qip-bench") if os.path.isdir("/content/qip-bench") else None
    if PROJECT is None:
        if SOURCE == "github":
            subprocess.run(["git", "clone", "-q", GITHUB_URL, "/content/qip-bench"], check=True)
        elif SOURCE == "drive":
            from google.colab import drive
            drive.mount("/content/drive")
            shutil.copytree(DRIVE_PATH, "/content/qip-bench", dirs_exist_ok=True)
        else:
            from google.colab import files
            print("Choose qip-bench.zip …")
            up = files.upload()
            name = next(iter(up))
            with zipfile.ZipFile(name) as z:
                z.extractall("/content/qip-bench")
            os.remove(name)
        PROJECT = find_project("/content/qip-bench")
else:                                   # running locally (e.g. Jupyter on the laptop)
    here = os.path.abspath(".")
    PROJECT = here if os.path.isdir(os.path.join(here, "qipbench")) else os.path.dirname(here)

assert PROJECT and os.path.isdir(os.path.join(PROJECT, "qipbench")), "project folder with qipbench/ not found"
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("✅ project at", PROJECT)
print("   contents:", ", ".join(sorted(p for p in os.listdir(PROJECT) if not p.startswith("."))))

In [ ]:
#@title 0.2 · Install quantum dependencies (≈ 1 min) { display-mode: "form" }
# Colab already ships numpy, scipy, pandas, matplotlib, scikit-image, OpenCV, plotly and ipywidgets.
# Versions are pinned to the ones the project was tested and benchmarked with.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "qiskit==2.5.1", "qiskit-aer==0.18.2", "pylatexenc",
                    "fastapi", "uvicorn", "python-multipart", "psutil"], check=True)
import qiskit, qiskit_aer, numpy, skimage, cv2, plotly, ipywidgets, psutil
print(f"✅ qiskit {qiskit.__version__} · qiskit-aer {qiskit_aer.__version__} · numpy {numpy.__version__} · "
      f"scikit-image {skimage.__version__} · OpenCV {cv2.__version__} · plotly {plotly.__version__}")
print(f"   CPU cores: {os.cpu_count()} · RAM free: {psutil.virtual_memory().available / 2**30:.1f} GB")

In [ ]:
#@title 0.3 · Initialise visual toolkit { display-mode: "form" }
import numpy as np
from IPython.display import display, Markdown, Image
from qipbench import viz, images, __version__
viz.setup()
print(f"✅ QIP-Bench {__version__} ready")

---
## 1 · Correctness first

Every quantum routine is checked against an exact classical answer:
* encode → decode round trips for all three encodings and both circuit structures
* the FRQI statevector equals its mathematical definition
* the Gray-code multiplexer is **operator-identical** to Qiskit's built-in `UCRYGate`
* the QHED decrement unitary, and QHED output equal to the classical pixel differences

In [ ]:
#@title 1 · Run the test suite { display-mode: "form" }
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", "tests"], capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

---
## 2 · The quantum image-processing pipeline

`classical preprocessing → quantum encoding → (operations) → measurement → reconstruction → evaluation`

**FRQI** stores pixel *i* in one colour qubit, $\cos\theta_i|0\rangle + \sin\theta_i|1\rangle$ with $\theta_i = \tfrac{\pi}{2}I_i$,
entangled with the position register $|i\rangle$. **NEQR** stores the grey level as a q-bit basis state $|C_i\rangle|i\rangle$.

Change the form values and re-run the cell to show other settings live.

In [ ]:
#@title 2.1 · Run one pipeline { display-mode: "form" }
ENCODING = "frqi"  #@param ["frqi", "neqr", "amplitude"]
IMAGE = "camera"  #@param ["camera", "phantom", "coins", "moon", "gradient", "checkerboard", "disk", "blocks"]
SIZE = 8  #@param [2, 4, 8, 16] {type:"raw"}
BIT_DEPTH = 8  #@param [1, 2, 4, 6, 8] {type:"raw"}
SHOTS = 4096  #@param [64, 256, 1024, 4096, 16384] {type:"raw"}
NOISE = 0.0  #@param [0.0, 0.0001, 0.001, 0.003, 0.01] {type:"raw"}
if ENCODING == "neqr":                    # NEQR qubits = 2·log2(size) + bit depth (+1 ancilla)
    new = (min(SIZE, 4), min(BIT_DEPTH, 4)) if NOISE else (min(SIZE, 8), BIT_DEPTH)
    if new != (SIZE, BIT_DEPTH):
        SIZE, BIT_DEPTH = new
        print(f"NEQR adjusted to {SIZE}x{SIZE}, {BIT_DEPTH}-bit to keep the simulation interactive")
res = viz.run(encoding=ENCODING, image=IMAGE, size=SIZE, bit_depth=BIT_DEPTH, shots=SHOTS, noise=NOISE)
viz.pipeline_strip(res)

In [ ]:
#@title 2.2 · FRQI vs NEQR vs amplitude encoding on the same image { display-mode: "form" }
cmp = {e: viz.run(encoding=e, image="phantom", size=4, bit_depth=8, shots=4096) for e in ("frqi", "neqr", "amplitude")}
display(viz.metrics_table(cmp))
display(Markdown("**Reading:** FRQI uses the fewest qubits but its quality depends on the shot count. NEQR needs more qubits and gates "
                 "but reads grey levels exactly. QPIE is the most qubit-efficient but needs the image norm as classical side information."))

---
## 3 · Seeing the quantum states

### 3.1 A qubit on the Bloch sphere

In [ ]:
viz.bloch_precession()

### 3.2 The circuit: baseline vs optimized structure (2×2 image, same quantum state)

In [ ]:
b = viz.run(encoding="frqi", image="camera", size=2, structure="baseline", shots=None)
o = viz.run(encoding="frqi", image="camera", size=2, structure="optimized", shots=None)
print(f"baseline: {b.metrics['cx']} CNOTs, depth {b.metrics['depth']}  |  optimized (Gray-code): {o.metrics['cx']} CNOTs, depth {o.metrics['depth']}")
viz.draw_circuit(o.circuit);

### 3.3 Every pixel is a qubit: animated scan (press ▶)

In [ ]:
viz.pixel_scan(viz.run(encoding='frqi', image='camera', size=4, shots=None))

### 3.4 Interactive: drag the slider to pick a pixel, and drag the sphere to rotate it

In [ ]:
viz.pixel_bloch(viz.run(encoding='frqi', image='camera', size=8, shots=4096))

---
## 4 · Measurement: watching the wavefunction collapse

The image is never read out directly. Each shot collapses the register to one basis state $|c\rangle|i\rangle$.
The animation replays the **actual measured shots** and decodes the image after each batch.
Blue cells have not been observed yet.

In [ ]:
#@title 4.1 · FRQI 8×8: quality grows with shots { display-mode: "form" }
viz.shot_replay(viz.run(encoding="frqi", image="camera", size=8, shots=4096))

In [ ]:
#@title 4.2 · NEQR 4×4 under noise p = 10⁻³: exact as soon as every pixel is seen { display-mode: "form" }
viz.shot_replay(viz.run(encoding="neqr", image="camera", size=4, bit_depth=4, shots=2048, noise=1e-3))

---
## 5 · Controlled factor experiments

Each animation changes **one** factor while every other factor stays fixed. This is the experimental method behind the research answer.

In [ ]:
#@title 5.1 · Noise: FRQI degrades gracefully, NEQR is perfect then collapses { display-mode: "form" }
viz.noise_sweep(image="camera", size=4, bit_depth=4)

In [ ]:
#@title 5.2 · Circuit structure: identical state, very different cost { display-mode: "form" }
viz.structure_comparison(image="camera", size=4, noise=1e-3)

In [ ]:
#@title 5.3 · Pick any factor to animate { display-mode: "form" }
FACTOR = "shots"  #@param ["shots", "size", "bit_depth", "operations", "interpolation", "normalization", "opt_level"]
ENC = "frqi"  #@param ["frqi", "neqr"]
LEVELS = {
    "shots": [32, 128, 512, 2048, 8192, 32768], "size": [2, 4, 8, 16], "bit_depth": [1, 2, 3, 4, 6, 8],
    "operations": [0, 1, 2, 4, 8, 16], "interpolation": ["nearest", "bilinear", "bicubic", "area"],
    "normalization": ["none", "minmax", "equalize"], "opt_level": [0, 1, 2, 3],
}[FACTOR]
base = dict(size=8, shots=4096) if ENC == "frqi" else dict(size=4, bit_depth=4, shots=4096)
if FACTOR in ("operations", "opt_level"):
    base.update(size=4, noise=2e-3, structure="baseline" if FACTOR == "opt_level" else "optimized")
if ENC == "neqr" and FACTOR == "size":
    LEVELS = [2, 4, 8]
base.pop(FACTOR, None)
viz.factor_sweep(FACTOR, LEVELS, encoding=ENC, image="camera", **base)

---
## 6 · Quantum Hadamard Edge Detection (QHED)

The image is amplitude-encoded with one ancilla, then $H \cdot D \cdot H$ is applied, where $D$ is a cyclic decrement. After this, every odd basis state
carries a neighbour difference $(c_i - c_{i+1})/2$. Edges appear as the measurement statistics accumulate.

In [ ]:
#@title 6 · QHED on a 16×16 image { display-mode: "form" }
EDGE_IMAGE = "blocks"  #@param ["blocks", "phantom", "camera", "checkerboard", "disk"]
viz.qhed_animation(image=EDGE_IMAGE, size=16, threshold=0.2 if EDGE_IMAGE in ("blocks", "checkerboard", "disk") else 0.12)

---
## 7 · The research answer: what really determines performance?

These results come from **≈ 2 500 simulated pipeline runs**: 12 one-factor-at-a-time sweeps plus a full 2⁷ factorial design
for each encoding, with main effects, interactions and p-values. Use the dropdowns, and hover for effect sizes.

In [ ]:
#@title 7.1 · Findings (auto-generated from the measured data) { display-mode: "form" }
report = open(os.path.join(PROJECT, "results", "REPORT.md"), encoding="utf-8").read()
display(Markdown(report.split("## 3.")[0]))

In [ ]:
#@title 7.2 · Interactive factor ranking { display-mode: "form" }
summary = viz.load_summary()
viz.factor_ranking(summary)

In [ ]:
#@title 7.3 · Interactive response curves for every study { display-mode: "form" }
METRIC = "psnr"  #@param ["psnr", "ssim", "e2e_psnr", "cx", "depth", "qubits", "success", "coverage", "total_s"]
viz.ofat_explorer(summary, metric=METRIC)

In [ ]:
#@title 7.4 · (Optional) Re-run the experiments live { display-mode: "form" }
RERUN = "skip"  #@param ["skip", "quick (≈ 5 min on Colab)", "full (≈ 45 min on Colab)"]
if RERUN != "skip":
    out = os.path.join(PROJECT, "results_colab")
    cmd = [sys.executable, "-m", "qipbench", "run", "--out", out] + (["--quick"] if RERUN.startswith("quick") else [])
    print("running:", " ".join(cmd[2:]))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-2500:] or proc.stderr[-2500:])
    display(Markdown(open(os.path.join(out, "REPORT.md"), encoding="utf-8").read().split("## 3.")[0]))
    display(Image(os.path.join(out, "figures", "factorial_pareto_frqi.png")))
else:
    print("Using the saved full-study results in results/ (choose 'quick' to re-run live into results_colab/).")

---
## 8 · The full web dashboard, inside Colab

The FastAPI dashboard starts in the background and opens below. Its tabs are Overview, Quantum Lab with live shot replay and qubit explorer,
Compare Encodings, Edge Detection, Research Explorer and Methodology. Use **the link** to open it full-screen in a new tab.

In [ ]:
#@title 8 · Launch the dashboard { display-mode: "form" }
import time, urllib.request
PORT = 8000
log = open(os.path.join(PROJECT, "dashboard.log"), "w")
server = subprocess.Popen([sys.executable, "-m", "qipbench", "serve", "--host", "0.0.0.0", "--port", str(PORT)],
                          cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/meta", timeout=1); break
    except Exception:
        time.sleep(0.5)
else:
    print(open(os.path.join(PROJECT, "dashboard.log")).read()[-2000:])
    raise RuntimeError("dashboard did not start")
print("✅ dashboard running")
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, anchor_text="🔗 Open the QIP-Bench dashboard in a new tab")
    output.serve_kernel_port_as_iframe(PORT, height=900)
else:
    print(f"open http://127.0.0.1:{PORT}")

---
## 9 · Live control panel

Ask the faculty for any combination of factors. Set them and press **Run quantum pipeline**.

In [ ]:
viz.lab()

---
### Project structure
```
qipbench/   images · encoders (FRQI, NEQR, QPIE) · backend (Aer, noise) · pipeline · edge (QHED)
            metrics · studies (OFAT + 2^7 factorial) · analysis · report · server · viz
web/        dashboard (FastAPI + HTML/JS)        notebooks/  01 encoding · 02 edges · 03 analysis · 04 visual lab
results/    measured data, figures, REPORT.md    tests/      27 correctness tests
docs/       METHODOLOGY · PROJECT_CHARTER · COLAB_DEMO_GUIDE · SUBMISSION_TEMPLATES
```

In [ ]:
#@title Stop the dashboard (optional) { display-mode: "form" }
try:
    server.terminate(); print("dashboard stopped")
except NameError:
    print("dashboard was not started")